# 06.3 — Estudo B: Tipo de Gestação (GUTT/GURT/GGUT/etc.)

Análises causais baseadas no tipo de gestação + forest plot consolidado
A+B + resumo completo.

Estudos:
  B1 — GURT vs GUTT → Retorno 30d
  B2 — Gemelar (GG) vs Única (GU) → Retorno 30d
  B3 — Pré-Termo (GUPT/GGPT) → Duração Internação
  B4 — GUPT (pré-termo única) vs GUTT → Retorno 30d [NOVO]
  B5 — Multi-target: Ret90d para estudos principais [NOVO]

Método: AIPW com 5-fold cross-fitting (duplamente robusto)

In [ ]:
from __future__ import annotations
import json
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as scipy_stats
from sklearn.compose import ColumnTransformer
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import SimpleImputer, IterativeImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.model_selection import KFold

warnings.filterwarnings("ignore")

# ── Idioma / Language ─────────────────────────────────────────────────────────
LANG = "pt"  # "pt" | "en"  ← única linha a alterar para mudar idioma

_T = {
    "pt": {
        "heatmap_title":     "Heatmap: Tipo de Gestação × Outcomes (valores brutos anotados)",
        "heatmap_ylabel":    "Tipo de Gestação",
        "fp_xlabel":         "ATE (pp = pontos percentuais | h = horas)",
        "fp_title":          "Forest Plot Consolidado — Estudos A + B (AIPW 5-fold Cross-Fitting)",
        "fp_leg_a":          "Estudo A — Notação G/P/A",
        "fp_leg_b":          "Estudo B — Tipo Gestação",
    },
    "en": {
        "heatmap_title":     "Heatmap: Pregnancy Type × Outcomes (raw annotated values)",
        "heatmap_ylabel":    "Pregnancy Type",
        "fp_xlabel":         "ATE (pp = percentage points | h = hours)",
        "fp_title":          "Consolidated Forest Plot — Studies A + B (AIPW 5-fold Cross-Fitting)",
        "fp_leg_a":          "Study A — Obstetric Notation G/P/A",
        "fp_leg_b":          "Study B — Pregnancy Type",
    },
}
T = _T[LANG]

ROOT    = Path("..").resolve()
ART_DIR = ROOT / "out" / "artifacts_causal_gestacao"
OUT_DIR = ROOT / "out" / "06.3_estudo_b"
OUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Carregar Artefatos de 06.1

In [ ]:
with open(ART_DIR / "data_gestacional.pkl", "rb") as f:
    _art = pickle.load(f)

df               = _art["df"]
CONFUNDIDORES_BASE = _art["CONFUNDIDORES_BASE"]
top_tipos        = _art["top_tipos"]

print(f"Dados carregados: {len(df):,} registros")
print(f"Tipos disponíveis: {df['TIPO_GESTACAO'].value_counts().head(10).to_dict()}")

## 2. Funções Auxiliares (standalone)

In [ ]:
def make_preprocessor(X):
    """Preprocessador com MICE para numérico e most_frequent para categórico."""
    num = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    cat = [c for c in X.columns if c not in num]
    transformers = []
    if num:
        transformers.append((
            "num",
            Pipeline([
                ("imp", IterativeImputer(max_iter=10, random_state=42)),
                ("sc",  StandardScaler()),
            ]),
            num,
        ))
    if cat:
        transformers.append((
            "cat",
            Pipeline([
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
            ]),
            cat,
        ))
    return ColumnTransformer(transformers, remainder="drop")


def aipw_binario(df_ana, T_col, Y_col, confundidores, n_boot=500):
    """AIPW com 5-fold cross-fitting para outcome binário."""
    df_a = df_ana[[T_col, Y_col] + [c for c in confundidores if c in df_ana.columns]].copy()
    df_a = df_a.dropna(subset=[T_col, Y_col])
    df_a = df_a[df_a[Y_col].isin([0, 1])]
    if len(df_a) < 100:
        return None

    T = df_a[T_col].values.astype(int)
    Y = df_a[Y_col].values.astype(int)
    CONF = [c for c in confundidores if c in df_a.columns]

    pre = make_preprocessor(df_a[CONF])
    X = pre.fit_transform(df_a[CONF])

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    sc1 = np.zeros(len(Y))
    sc0 = np.zeros(len(Y))

    for tr_idx, te_idx in kf.split(X):
        ps_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        ps_f.fit(X[tr_idx], T[tr_idx])
        ps_te = np.clip(ps_f.predict_proba(X[te_idx])[:, 1], 0.05, 0.95)

        X_out_tr = np.hstack([X[tr_idx], T[tr_idx].reshape(-1, 1)])
        out_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        out_f.fit(X_out_tr, Y[tr_idx])

        X1_te = np.hstack([X[te_idx], np.ones((len(te_idx), 1))])
        X0_te = np.hstack([X[te_idx], np.zeros((len(te_idx), 1))])
        mu1_te = out_f.predict_proba(X1_te)[:, 1]
        mu0_te = out_f.predict_proba(X0_te)[:, 1]

        sc1[te_idx] = mu1_te + T[te_idx] * (Y[te_idx] - mu1_te) / (ps_te + 1e-9)
        sc0[te_idx] = mu0_te + (1 - T[te_idx]) * (Y[te_idx] - mu0_te) / (1 - ps_te + 1e-9)

    ate = (sc1 - sc0).mean()
    rng = np.random.default_rng(42)
    boots = [
        (sc1[rng.integers(0, len(Y), len(Y))] - sc0[rng.integers(0, len(Y), len(Y))]).mean()
        for _ in range(n_boot)
    ]
    se = np.std(boots)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    pval = 2 * (1 - scipy_stats.norm.cdf(abs(ate / se))) if se > 0 else 1.0

    return {
        "n": len(df_a),
        "n_exp": int(T.sum()),
        "prev_Y": round(Y.mean() * 100, 2),
        "ate_pp": round(ate * 100, 4),
        "ic_lo_pp": round(lo * 100, 4),
        "ic_hi_pp": round(hi * 100, 4),
        "se": round(se, 4),
        "p_valor": round(pval, 4),
    }


def aipw_continuo(df_ana, T_col, Y_col, confundidores, n_boot=500):
    """AIPW com 5-fold cross-fitting para outcome contínuo."""
    df_a = df_ana[[T_col, Y_col] + [c for c in confundidores if c in df_ana.columns]].copy()
    df_a = df_a.dropna(subset=[T_col, Y_col])
    df_a = df_a[df_a[Y_col].between(1, 720)]
    if len(df_a) < 100:
        return None

    T = df_a[T_col].values.astype(int)
    Y = df_a[Y_col].values.astype(float)
    CONF = [c for c in confundidores if c in df_a.columns]

    pre = make_preprocessor(df_a[CONF])
    X = pre.fit_transform(df_a[CONF])

    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    sc1 = np.zeros(len(Y))
    sc0 = np.zeros(len(Y))

    for tr_idx, te_idx in kf.split(X):
        ps_f = GradientBoostingClassifier(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        ps_f.fit(X[tr_idx], T[tr_idx])
        ps_te = np.clip(ps_f.predict_proba(X[te_idx])[:, 1], 0.05, 0.95)

        X_out_tr = np.hstack([X[tr_idx], T[tr_idx].reshape(-1, 1)])
        out_f = GradientBoostingRegressor(
            n_estimators=200, max_depth=3, learning_rate=0.05, random_state=42
        )
        out_f.fit(X_out_tr, Y[tr_idx])

        X1_te = np.hstack([X[te_idx], np.ones((len(te_idx), 1))])
        X0_te = np.hstack([X[te_idx], np.zeros((len(te_idx), 1))])
        mu1_te = out_f.predict(X1_te)
        mu0_te = out_f.predict(X0_te)

        sc1[te_idx] = mu1_te + T[te_idx] * (Y[te_idx] - mu1_te) / (ps_te + 1e-9)
        sc0[te_idx] = mu0_te + (1 - T[te_idx]) * (Y[te_idx] - mu0_te) / (1 - ps_te + 1e-9)

    ate = (sc1 - sc0).mean()
    rng = np.random.default_rng(42)
    boots = [
        (sc1[rng.integers(0, len(Y), len(Y))] - sc0[rng.integers(0, len(Y), len(Y))]).mean()
        for _ in range(n_boot)
    ]
    se = np.std(boots)
    lo, hi = np.percentile(boots, [2.5, 97.5])
    pval = 2 * (1 - scipy_stats.norm.cdf(abs(ate / se))) if se > 0 else 1.0

    return {
        "n": len(df_a),
        "n_exp": int(T.sum()),
        "Y_media_exp":  round(float(Y[T == 1].mean()), 1),
        "Y_media_ctrl": round(float(Y[T == 0].mean()), 1),
        "ate_h":   round(ate, 2),
        "ic_lo_h": round(lo, 2),
        "ic_hi_h": round(hi, 2),
        "se":      round(se, 4),
        "p_valor": round(pval, 4),
    }


def e_value_risk_diff(ate_pp: float, baseline_risk_pct: float):
    """E-value para diferença de risco (VanderWeele & Ding, 2017)."""
    p0 = baseline_risk_pct / 100
    p1 = p0 + ate_pp / 100
    if p1 <= 0 or p0 <= 0:
        return np.nan, np.nan
    rr = max(p1 / p0, p0 / p1)
    ev = rr + np.sqrt(rr * (rr - 1))
    return round(rr, 3), round(ev, 3)


def smd(df_local, col, T_col="T"):
    """Standardized Mean Difference para avaliação de balanço."""
    g1 = pd.to_numeric(df_local[df_local[T_col] == 1][col], errors="coerce").dropna()
    g0 = pd.to_numeric(df_local[df_local[T_col] == 0][col], errors="coerce").dropna()
    if len(g1) == 0 or len(g0) == 0:
        return np.nan
    pooled = np.sqrt((g1.std() ** 2 + g0.std() ** 2) / 2)
    return abs(g1.mean() - g0.mean()) / pooled if pooled > 0 else 0.0


def _print_binario(nome, r):
    if r is None:
        print("  Amostra insuficiente (<100).")
        return
    print(f"  N análise:      {r['n']:,}")
    print(f"  N expostos:     {r['n_exp']:,} ({r['n_exp']/r['n']*100:.1f}%)")
    print(f"  Prevalência Y:  {r['prev_Y']}%")
    print(f"  ATE (AIPW):    {r['ate_pp']:+.2f} pp")
    print(f"  IC95%:         [{r['ic_lo_pp']:+.2f}, {r['ic_hi_pp']:+.2f}]")
    print(f"  p-valor:       {r['p_valor']:.4f}")
    rr, ev = e_value_risk_diff(r["ate_pp"], r["prev_Y"])
    print(f"  E-value:       RR_obs={rr}  E={ev}")


def _print_continuo(nome, r):
    if r is None:
        print("  Amostra insuficiente.")
        return
    print(f"  Duração — expostos: {r['Y_media_exp']:.1f}h  vs  controles: {r['Y_media_ctrl']:.1f}h")
    print(f"  ATE (AIPW):    {r['ate_h']:+.1f}h")
    print(f"  IC95%:         [{r['ic_lo_h']:+.1f}, {r['ic_hi_h']:+.1f}]")
    print(f"  p-valor:       {r['p_valor']:.4f}")

## 3. Estudo B1 — GURT vs GUTT → Retorno 30d

In [ ]:
print("=" * 60)
print("ESTUDO B1 — GURT vs GUTT → Retorno 30d")
print("(Gestação de Risco vs Gestação Normal a Termo)")
print("=" * 60)

df_tipo = df[df["TIPO_GESTACAO"].isin(["GURT", "GUTT", "GUR", "GUPT"])].copy()
df_tipo["T_risco"] = df_tipo["TIPO_GESTACAO"].isin(["GURT", "GUR"]).astype(int)
conf_b1 = [c for c in CONFUNDIDORES_BASE if c in df_tipo.columns]

if len(df_tipo) > 100:
    r_b1 = aipw_binario(df_tipo, "T_risco", "Y_ret30", conf_b1)
    _print_binario("B1", r_b1)
else:
    r_b1 = None
    print("  Amostra insuficiente (<100). Verifique cobertura do texto.")

## 4. Estudo B2 — Gemelar (GG) vs Única (GU) → Retorno 30d

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO B2 — Gestação Gemelar (GG) vs Única (GU) → Retorno 30d")
print("=" * 60)

df_mult = df[df["TIPO_GESTACAO"].str.startswith(("GG", "GU"), na=False)].copy()
df_mult["T_gemelar"] = df_mult["TIPO_GESTACAO"].str.startswith("GG").astype(int)
conf_b2 = [c for c in CONFUNDIDORES_BASE if c in df_mult.columns]

if len(df_mult) > 100 and df_mult["T_gemelar"].sum() > 20:
    r_b2 = aipw_binario(df_mult, "T_gemelar", "Y_ret30", conf_b2)
    _print_binario("B2", r_b2)
else:
    r_b2 = None
    print("  Amostra insuficiente.")

## 5. Estudo B3 — Pré-Termo → Duração Internação

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO B3 — Gestação Pré-Termo → Duração Internação (horas)")
print("=" * 60)

df_pt = df[df["TIPO_GESTACAO"].isin(
    ["GUPT", "GGPT", "PRETERMO_SEM_MULT", "GUTT", "GGUT"]
)].copy()
df_pt["T_pretermo"] = df_pt["TIPO_GESTACAO"].isin(
    ["GUPT", "GGPT", "PRETERMO_SEM_MULT"]
).astype(int)
df_pt = df_pt[df_pt["Y_dur"].notna()]
conf_b3 = [c for c in CONFUNDIDORES_BASE if c in df_pt.columns]

if len(df_pt) > 100 and df_pt["T_pretermo"].sum() > 20:
    r_b3 = aipw_continuo(df_pt, "T_pretermo", "Y_dur", conf_b3)
    _print_continuo("B3", r_b3)
else:
    r_b3 = None
    print("  Amostra insuficiente.")

## 6. Estudo B4 — GUPT vs GUTT → Retorno 30d (NOVO)

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO B4 — GUPT (Pré-Termo Única) vs GUTT → Retorno 30d")
print("=" * 60)

df_b4 = df[df["TIPO_GESTACAO"].isin(["GUPT", "GUTT"])].copy()
df_b4["T_gupt"] = (df_b4["TIPO_GESTACAO"] == "GUPT").astype(int)
conf_b4 = [c for c in CONFUNDIDORES_BASE if c in df_b4.columns]

if len(df_b4) > 100 and df_b4["T_gupt"].sum() > 20:
    r_b4 = aipw_binario(df_b4, "T_gupt", "Y_ret30", conf_b4)
    _print_binario("B4", r_b4)
else:
    r_b4 = None
    print("  Amostra insuficiente.")

## 7. Estudo B5 — Multi-target: Retorno 90d (NOVO)

In [ ]:
print("\n" + "=" * 60)
print("ESTUDO B5 — Multi-target: Retorno 90d nos Estudos B Principais")
print("=" * 60)

r_b5_b1 = None
r_b5_b2 = None

if "Y_ret90" in df.columns and df["Y_ret90"].notna().sum() > 100:
    # B5a: GURT → Ret90d
    print("\n  B5a — GURT vs GUTT → Retorno 90d")
    df_b5a = df[df["TIPO_GESTACAO"].isin(["GURT", "GUTT", "GUR"])].copy()
    df_b5a["T_risco"] = df_b5a["TIPO_GESTACAO"].isin(["GURT", "GUR"]).astype(int)
    conf_b5a = [c for c in CONFUNDIDORES_BASE if c in df_b5a.columns]
    if len(df_b5a) > 100:
        r_b5_b1 = aipw_binario(df_b5a, "T_risco", "Y_ret90", conf_b5a)
        _print_binario("B5a", r_b5_b1)
    else:
        print("  Amostra insuficiente.")

    # B5b: Gemelar → Ret90d
    print("\n  B5b — Gemelar (GG) vs Única (GU) → Retorno 90d")
    df_b5b = df[df["TIPO_GESTACAO"].str.startswith(("GG", "GU"), na=False)].copy()
    df_b5b["T_gemelar"] = df_b5b["TIPO_GESTACAO"].str.startswith("GG").astype(int)
    conf_b5b = [c for c in CONFUNDIDORES_BASE if c in df_b5b.columns]
    if len(df_b5b) > 100 and df_b5b["T_gemelar"].sum() > 20:
        r_b5_b2 = aipw_binario(df_b5b, "T_gemelar", "Y_ret90", conf_b5b)
        _print_binario("B5b", r_b5_b2)
    else:
        print("  Amostra insuficiente.")
else:
    print("  Y_ret90 não disponível ou insuficiente.")

## 8. Heatmap: tipo_gestacao × outcome

In [ ]:
print("\n" + "=" * 60)
print("HEATMAP — Tipo de Gestação × Outcomes")
print("=" * 60)

heat_tipos = [t for t in top_tipos if t != "NAO_IDENTIFICADO"]
df_heat_raw = df[df["TIPO_GESTACAO"].isin(heat_tipos)].copy()

heat_rows = []
for tipo in heat_tipos:
    sub = df_heat_raw[df_heat_raw["TIPO_GESTACAO"] == tipo]
    if len(sub) < 10:
        continue
    row = {
        "tipo_gestacao": tipo,
        "n": len(sub),
        "retorno_30d_pct": sub["Y_ret30"].mean() * 100,
        "duracao_media_h": sub["Y_dur"].mean() if "Y_dur" in sub else np.nan,
    }
    if "Y_ret90" in sub.columns:
        row["retorno_90d_pct"] = sub["Y_ret90"].mean() * 100
    heat_rows.append(row)

df_heat = pd.DataFrame(heat_rows).set_index("tipo_gestacao")
df_heat.to_csv(OUT_DIR / "heatmap_tipo_outcome.csv")

# Plot heatmap
heat_cols = [c for c in ["retorno_30d_pct", "retorno_90d_pct", "duracao_media_h"] if c in df_heat.columns]
if len(heat_cols) >= 1 and len(df_heat) >= 2:
    fig, ax = plt.subplots(figsize=(max(6, len(heat_cols) * 2), max(4, len(df_heat) * 0.7)))
    heat_data = df_heat[heat_cols].copy()
    # Normaliza por coluna para visualização comparável
    heat_norm = (heat_data - heat_data.min()) / (heat_data.max() - heat_data.min() + 1e-9)
    sns.heatmap(
        heat_norm,
        annot=heat_data.round(1),
        fmt=".1f",
        cmap="RdYlGn_r",
        ax=ax,
        linewidths=0.5,
        cbar_kws={"label": "Valor normalizado [0-1]"},
    )
    ax.set_title(T["heatmap_title"])
    ax.set_xlabel("")
    ax.set_ylabel(T["heatmap_ylabel"])
    plt.tight_layout()
    plt.savefig(OUT_DIR / "heatmap_tipo_outcome.png", dpi=150)
    plt.show()
    plt.close()
    print(f"  Heatmap salvo: {OUT_DIR / 'heatmap_tipo_outcome.png'}")
else:
    print("  Dados insuficientes para heatmap.")

## 9. Forest Plot Consolidado A + B

In [ ]:
# Carrega resultados do Estudo A
_a_path = ART_DIR / "resultados_estudo_a.json"
resultados_a_ext = {}
if _a_path.exists():
    with open(_a_path, "r", encoding="utf-8") as f:
        resultados_a_ext = json.load(f)
    print(f"Resultados A carregados de: {_a_path}")
else:
    print(f"[AVISO] {_a_path} não encontrado — execute 06.2 primeiro")

# Mapeia resultados A para lista de tuplas
_estudos_a_consolidado = [
    ("A1 — Grande Multípara → Ret30d",    resultados_a_ext.get("A1_grande_multipara_ret30d"), "pp"),
    ("A2 — Aborto Anterior → Ret30d",      resultados_a_ext.get("A2_aborto_anterior_ret30d"),  "pp"),
    ("A3 — Grande Multípara → Duração",    resultados_a_ext.get("A3_grande_multipara_duracao"), "h"),
    ("A4 — G>=5 vs G<=4 → Ret30d",         resultados_a_ext.get("A4_dose_resposta_G5_ret30d"),  "pp"),
    ("A5 — Aborto Recorrente → Ret30d",    resultados_a_ext.get("A5_aborto_recorrente_ret30d"), "pp"),
]

_estudos_b_consolidado = [
    ("B1 — GURT vs GUTT → Ret30d",  r_b1, "pp"),
    ("B2 — Gemelar → Ret30d",        r_b2, "pp"),
    ("B3 — Pré-Termo → Duração",     r_b3, "h"),
    ("B4 — GUPT vs GUTT → Ret30d",   r_b4, "pp"),
]

todos_estudos = [
    (lbl, r, unit)
    for lbl, r, unit in (_estudos_a_consolidado + _estudos_b_consolidado)
    if r is not None
]

fig, ax = plt.subplots(figsize=(12, max(5, len(todos_estudos) * 1.3)))
y_pos = range(len(todos_estudos))
cores_grupo = ["#1f77b4"] * len([e for e in _estudos_a_consolidado if e[1] is not None]) + \
              ["#ff7f0e"] * len([e for e in _estudos_b_consolidado if e[1] is not None])

for i, (lbl, r, unit) in enumerate(todos_estudos):
    # Corrige o bug 0.0: usa .get() com verificação explícita de None
    ate = r.get("ate_pp") if r.get("ate_pp") is not None else r.get("ate_h", 0)
    lo  = r.get("ic_lo_pp") if r.get("ic_lo_pp") is not None else r.get("ic_lo_h", 0)
    hi  = r.get("ic_hi_pp") if r.get("ic_hi_pp") is not None else r.get("ic_hi_h", 0)
    cor = cores_grupo[i] if i < len(cores_grupo) else "#9467bd"
    ax.errorbar(
        ate, i,
        xerr=[[ate - lo], [hi - ate]],
        fmt="o", color=cor, capsize=5, markersize=7,
    )
    p = r["p_valor"]
    sig = "***" if p < 0.001 else "**" if p < 0.01 else "*" if p < 0.05 else "ns"
    ax.text(hi + 0.3, i, f"p={p:.3f} {sig}", va="center", fontsize=8)

ax.axvline(0, color="black", lw=0.8, ls="--")
ax.set_yticks(list(y_pos))
ax.set_yticklabels([e[0] for e in todos_estudos], fontsize=9)
ax.set_xlabel(T["fp_xlabel"])
ax.set_title(T["fp_title"])
ax.grid(axis="x", alpha=0.3)

# Legenda de grupos
from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#1f77b4", markersize=9, label=T["fp_leg_a"]),
    Line2D([0], [0], marker="o", color="w", markerfacecolor="#ff7f0e", markersize=9, label=T["fp_leg_b"]),
]
ax.legend(handles=legend_elements, loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(OUT_DIR / "forest_plot_consolidado_ab.png", dpi=150)
plt.show()
plt.close()

## 10. Resumo Final e Exportação

In [ ]:
resultados_b = {
    "B1_GURT_vs_GUTT_ret30d":    r_b1,
    "B2_gemelar_ret30d":          r_b2,
    "B3_pretermo_duracao":        r_b3,
    "B4_GUPT_vs_GUTT_ret30d":    r_b4,
    "B5a_GURT_ret90d":            r_b5_b1,
    "B5b_gemelar_ret90d":         r_b5_b2,
}

resumo_completo = {
    "data": __import__("datetime").datetime.now().isoformat(),
    "estudos_A": resultados_a_ext,
    "estudos_B": resultados_b,
}

(OUT_DIR / "resumo_causal_gestacao_completo.json").write_text(
    json.dumps(resumo_completo, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)
(ART_DIR / "resumo_causal_gestacao_completo.json").write_text(
    json.dumps(resumo_completo, ensure_ascii=False, indent=2, default=str),
    encoding="utf-8",
)

## Resumo 06.3

In [ ]:
print("\n" + "=" * 60)
print("╔══ 06 Causal Gestacional — CONCLUIDO ══╗")
print("=" * 60)
print("\n  === ESTUDO A — Notacao Obstetrica (G/P/A) ===")
for label, r, unit in _estudos_a_consolidado:
    if r is None:
        continue
    ate = r.get("ate_pp") if r.get("ate_pp") is not None else r.get("ate_h", 0)
    lo  = r.get("ic_lo_pp") if r.get("ic_lo_pp") is not None else r.get("ic_lo_h", 0)
    hi  = r.get("ic_hi_pp") if r.get("ic_hi_pp") is not None else r.get("ic_hi_h", 0)
    p   = r["p_valor"]
    sig = "SIGNIFICATIVO" if p < 0.05 else "nao significativo"
    rr, ev = e_value_risk_diff(
        r.get("ate_pp", 0.0) if r.get("ate_pp") is not None else 0.0,
        r.get("prev_Y", 0.0) if r.get("prev_Y") is not None else 0.0,
    )
    print(f"\n  {label}")
    print(f"  ATE={ate:+.2f} {unit}  IC95%[{lo:+.2f},{hi:+.2f}]  p={p:.4f}  E-value={ev}  -> {sig}")

print("\n  === ESTUDO B — Tipo de Gestacao ===")
estudos_b_print = [
    ("B1 — GURT vs GUTT → Ret30d",  r_b1, "pp"),
    ("B2 — Gemelar → Ret30d",        r_b2, "pp"),
    ("B3 — Pre-Termo → Duracao",     r_b3, "h"),
    ("B4 — GUPT vs GUTT → Ret30d",   r_b4, "pp"),
    ("B5a — GURT → Ret90d",          r_b5_b1, "pp"),
    ("B5b — Gemelar → Ret90d",       r_b5_b2, "pp"),
]
for label, r, unit in estudos_b_print:
    if r is None:
        continue
    ate = r.get("ate_pp") if r.get("ate_pp") is not None else r.get("ate_h", 0)
    lo  = r.get("ic_lo_pp") if r.get("ic_lo_pp") is not None else r.get("ic_lo_h", 0)
    hi  = r.get("ic_hi_pp") if r.get("ic_hi_pp") is not None else r.get("ic_hi_h", 0)
    p   = r["p_valor"]
    sig = "SIGNIFICATIVO" if p < 0.05 else "nao significativo"
    rr, ev = e_value_risk_diff(
        r.get("ate_pp", 0.0) if r.get("ate_pp") is not None else 0.0,
        r.get("prev_Y", 0.0) if r.get("prev_Y") is not None else 0.0,
    )
    print(f"\n  {label}")
    print(f"  ATE={ate:+.2f} {unit}  IC95%[{lo:+.2f},{hi:+.2f}]  p={p:.4f}  E-value={ev}  -> {sig}")

print("\n" + "=" * 60)
print(f"  Outputs: {OUT_DIR}")
print(f"  JSON completo: {OUT_DIR / 'resumo_causal_gestacao_completo.json'}")
print(f"  Forest consolidado: {OUT_DIR / 'forest_plot_consolidado_ab.png'}")
print(f"  Heatmap: {OUT_DIR / 'heatmap_tipo_outcome.png'}")
print("=" * 60)